# CLRKDNet ONNX Export and Smoke Test

Use this notebook in Colab to validate the lane-model deployment path before map-based data collection.

It checks: repo import, checkpoint load, PyTorch raw forward, ONNX export, and ONNX Runtime output difference.

Upload or clone `CLRKDNet` to `/content/CLRKDNet`, and upload `ResNet18_CULane.pth` to `/content/ResNet18_CULane.pth`.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/content')
CLRKDNET_DIR = PROJECT_ROOT / 'CLRKDNet'
CONFIG_PATH = CLRKDNET_DIR / 'configs' / 'ResNet18_CULane.py'
CHECKPOINT_PATH = PROJECT_ROOT / 'ResNet18_CULane.pth'
ONNX_PATH = PROJECT_ROOT / 'clrkdnet_resnet18_culane.onnx'
SAMPLE_IMAGE_PATH = None  # Example: Path('/content/sample.jpg')

USE_FAKE_NMS_FOR_EXPORT_ONLY = True

print('CLRKDNet dir:', CLRKDNET_DIR)
print('Config:', CONFIG_PATH)
print('Checkpoint:', CHECKPOINT_PATH)
print('ONNX output:', ONNX_PATH)

## Install dependencies

If `mmcv==1.2.5` fails in your Colab runtime, try another old 1.x version such as `mmcv==1.7.2`. The export path mainly needs `mmcv.cnn.ConvModule`.

In [ ]:
!pip -q install addict yapf pathspec timm opencv-python onnx onnxruntime onnxscript mmcv==1.2.5

In [ ]:
import os
import sys
import types
import numpy as np

assert CLRKDNET_DIR.exists(), f'Missing repo: {CLRKDNET_DIR}'
assert CONFIG_PATH.exists(), f'Missing config: {CONFIG_PATH}'
assert CHECKPOINT_PATH.exists(), f'Missing checkpoint: {CHECKPOINT_PATH}'

os.chdir(str(CLRKDNET_DIR))
if str(CLRKDNET_DIR) not in sys.path:
    sys.path.insert(0, str(CLRKDNET_DIR))

# CLRKDNet imports custom CUDA NMS at module import time, but raw forward export does not call NMS.
if USE_FAKE_NMS_FOR_EXPORT_ONLY:
    fake_nms_impl = types.ModuleType('clrkd.ops.nms_impl')
    def _nms_forward(*args, **kwargs):
        raise RuntimeError('NMS extension is unavailable. Raw ONNX export does not call NMS.')
    fake_nms_impl.nms_forward = _nms_forward
    sys.modules['clrkd.ops.nms_impl'] = fake_nms_impl

print('Repo path is ready')

In [ ]:
import torch

from clrkd.utils.config import Config
import clrkd.models  # Registers backbones, necks, heads, and nets.
from clrkd.models.registry import build_net

def clean_state_dict(raw):
    state = raw.get('net', raw) if isinstance(raw, dict) else raw
    cleaned = {}
    for key, value in state.items():
        cleaned[key[7:] if key.startswith('module.') else key] = value
    return cleaned

cfg = Config.fromfile(str(CONFIG_PATH))
cfg.backbone.pretrained = False
model = build_net(cfg)
checkpoint = torch.load(str(CHECKPOINT_PATH), map_location='cpu')
state = clean_state_dict(checkpoint)
load_result = model.load_state_dict(state, strict=False)
model.eval()

print('Config img size:', cfg.img_w, cfg.img_h)
print('Missing keys:', len(load_result.missing_keys))
print('Unexpected keys:', len(load_result.unexpected_keys))

In [ ]:
class RawForwardWrapper(torch.nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, image):
        output = self.net(image)
        if isinstance(output, (list, tuple)):
            output = output[-1]
        return output

wrapper = RawForwardWrapper(model).eval()
dummy = torch.randn(1, 3, cfg.img_h, cfg.img_w, dtype=torch.float32)

with torch.no_grad():
    pytorch_output = wrapper(dummy)

print('PyTorch output shape:', tuple(pytorch_output.shape))
print('PyTorch output dtype:', pytorch_output.dtype)
print('PyTorch output min/max:', float(pytorch_output.min()), float(pytorch_output.max()))

In [ ]:
torch.onnx.export(
    wrapper,
    dummy,
    str(ONNX_PATH),
    input_names=['image'],
    output_names=['raw_predictions'],
    dynamic_axes={'image': {0: 'batch'}, 'raw_predictions': {0: 'batch'}},
    opset_version=16,
    do_constant_folding=True,
    dynamo=False,
)

size_mb = ONNX_PATH.stat().st_size / (1024 * 1024)
print('Exported:', ONNX_PATH)
print(f'ONNX size: {size_mb:.2f} MB')

In [ ]:
import onnx
import onnxruntime as ort

onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)

session = ort.InferenceSession(str(ONNX_PATH), providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

ort_output = session.run([output_name], {input_name: dummy.numpy()})[0]
pt_output = pytorch_output.detach().cpu().numpy()
abs_diff = np.abs(pt_output - ort_output)

print('ONNX input:', input_name, session.get_inputs()[0].shape)
print('ONNX output:', output_name, session.get_outputs()[0].shape)
print('max_abs_diff:', float(abs_diff.max()))
print('mean_abs_diff:', float(abs_diff.mean()))

In [ ]:
# Optional image smoke test. This does not prove map performance.
# It only checks that an image can pass through the same preprocessing and ONNX session.
import cv2

def preprocess_for_clrkdnet(image_bgr, cfg):
    h, w = image_bgr.shape[:2]
    crop_top = int(round(h * cfg.cut_height / cfg.ori_img_h))
    crop_top = min(max(crop_top, 0), max(0, h - 1))
    cropped = image_bgr[crop_top:, :, :]
    resized = cv2.resize(cropped, (cfg.img_w, cfg.img_h), interpolation=cv2.INTER_LINEAR)
    tensor = resized.astype(np.float32).transpose(2, 0, 1)[None, ...]
    return tensor, resized, crop_top

if SAMPLE_IMAGE_PATH is None:
    print('SAMPLE_IMAGE_PATH is None. Skipping image smoke test.')
else:
    image = cv2.imread(str(SAMPLE_IMAGE_PATH))
    assert image is not None, f'Could not read {SAMPLE_IMAGE_PATH}'
    tensor, resized, crop_top = preprocess_for_clrkdnet(image, cfg)
    raw = session.run([output_name], {input_name: tensor})[0]
    scores = np.exp(raw[0, :, :2] - raw[0, :, :2].max(axis=1, keepdims=True))
    scores = scores[:, 1] / scores.sum(axis=1)
    print('crop_top:', crop_top)
    print('preprocessed shape:', resized.shape)
    print('raw shape:', raw.shape)
    print('top lane confidence scores:', np.sort(scores)[-10:][::-1])

## Record these results

- ONNX export success/failure
- ONNX file size
- Input and output shape
- max_abs_diff and mean_abs_diff
- Any dependency or NMS issue

If this passes, copy the `.onnx` file to the Pi and run `03_lane_onnx_bench.py`.